# 04 | Individual morphological features and category-level statistical summaries

**Purpose.** Reproduce the selected individual-feature comparisons and all morphological category charts from the original `analise features-.ipynb`, without repeating LDA and K-means modeling already contained in Notebook 03.

**Original tests retained:** one-way ANOVA with Tukey HSD for individual features; Welch's t-test against `Non-treated_1h` for category summaries. Category scores are means of `−log10(p)` values, **not** effect sizes. No multiple-testing adjustment was present in the original category summaries.

**Important:** The original category code selects features using substring matching. A feature may therefore contribute to more than one category (e.g., a nucleus/vesicle descriptor). The same matching rule is preserved.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind
from statsmodels.stats.multicomp import pairwise_tukeyhsd

INPUT_CSV = Path("../results/INPUT_WELL_PROFILES.csv")  # Set to Notebook 02 output.
OUTPUT_DIR = Path("../results/morphological_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GROUP_COL = "Metadata_compound_time"
CONTROL_GROUP = "Non-treated_1h"
GROUP_ORDER = [
    "Non-treated_1h",
    "AuNP_1h", "AuNP_4h", "AuNP_24h",
    "AuNP-and-healthy-serum_1h", "AuNP-and-healthy-serum_4h", "AuNP-and-healthy-serum_24h",
    "AuNP-cancer-and-cachexia-serum_1h", "AuNP-cancer-and-cachexia-serum_4h",
    "AuNP-cancer-and-cachexia-serum_24h",
]
# Original experimental palette, preserved in the same group order.
ORIGINAL_COLORS = [
    "#272323", "#1730F0", "#0a17a4", "#08024C", "#f79d7c",
    "#ef5946", "#FB0505", "#784f48", "#c93d28", "#651105",
]
PALETTE = dict(zip(GROUP_ORDER, ORIGINAL_COLORS))

## 1. Load the normalized well-level profiles

The source data are the negative-control-normalized, feature-selected well profiles from Notebook 02. This notebook does not refit the LDA model. Treatment/time labels are constructed only if they are not already present.

In [ ]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Set INPUT_CSV to the actual output of Notebook 02: {INPUT_CSV}")
df = pd.read_csv(INPUT_CSV)
if "Metadata_time" not in df and "Metadata_concentration_uM" in df:
    df = df.rename(columns={"Metadata_concentration_uM": "Metadata_time"})
if GROUP_COL not in df:
    df[GROUP_COL] = df["Metadata_compound"].astype(str) + "_" + df["Metadata_time"].astype(str)
df = df.loc[~df[GROUP_COL].isin(["Non-treated_4h", "Non-treated_24h"])].copy()
GROUPS_PRESENT = [g for g in GROUP_ORDER if g in df[GROUP_COL].unique()]
if CONTROL_GROUP not in GROUPS_PRESENT: raise ValueError("The untreated 1 h reference is missing.")
print(f"Loaded {len(df)} well profiles across {len(GROUPS_PRESENT)} treatment/time groups")

## 2. Compare selected individual features using ANOVA and Tukey HSD

The original notebook repeated essentially the same boxplot, overlaid observations, one-way ANOVA, and Tukey HSD calculation for several morphological descriptors. These operations are consolidated into one reusable function, without changing the statistical tests.

The two `Object_Number` descriptors are included for traceability, but are object identifiers in standard CellProfiler output and should not be interpreted as biological counts without checking their upstream definitions.

In [ ]:
from itertools import combinations
from string import ascii_lowercase, ascii_uppercase

SELECTED_FEATURES = [
    "Nuclei_Granularity_8_AOGFP",
    "Nuclei_Granularity_10_AOPI",
    "Nuclei_Intensity_IntegratedIntensityEdge_AOGFP",
    "Nuclei_Mean_Nucleoli_AreaShape_Zernike_4_0",
    "Nuclei_Number_Object_Number",
    "Nuclei_Mean_Nucleoli_Number_Object_Number",
    "Cells_Mean_Vesicles_AreaShape_Compactness",
]


def tukey_compact_letters(tukey, groups):
    """Compact letter display matching all Tukey HSD reject decisions.

    Groups sharing a letter have a non-significant Tukey comparison.
    Groups with no shared letters have a significant Tukey comparison.
    This does not establish equivalence between non-significant groups.
    """
    non_significant = {(g, g) for g in groups}
    for (g1, g2), reject in zip(combinations(sorted(tukey.groupsunique), 2), tukey.reject):
        # statsmodels' MultiComparison pair indices are in sorted group order.
        if not reject:
            non_significant.update({(g1, g2), (g2, g1)})

    # Find maximal cliques in the non-significance graph. Each clique gets
    # one letter; this preserves overlapping letter assignments (e.g. 'ab').
    cliques = []
    for size in range(1, len(groups) + 1):
        for subset in combinations(groups, size):
            if all((a, b) in non_significant for a, b in combinations(subset, 2)):
                members = set(subset)
                if not any(members < other for other in cliques):
                    cliques = [other for other in cliques if not other < members]
                    cliques.append(members)
    cliques.sort(key=lambda members: min(groups.index(g) for g in members))
    symbols = ascii_lowercase + ascii_uppercase
    if len(cliques) > len(symbols):
        raise ValueError("Too many letter groups for single-character labels.")
    return {g: "".join(symbols[i] for i, members in enumerate(cliques) if g in members)
            for g in groups}


def analyze_feature(feature):
    if feature not in df.columns:
        print(f"Skipped unavailable feature: {feature}")
        return None
    plot_data = df[[GROUP_COL, feature]].copy()
    plot_data[feature] = pd.to_numeric(plot_data[feature], errors="coerce")
    plot_data = plot_data.dropna()
    groups = [g for g in GROUP_ORDER if (plot_data[GROUP_COL] == g).sum() > 0]
    values = [plot_data.loc[plot_data[GROUP_COL] == g, feature].to_numpy() for g in groups]
    if len(groups) < 2 or min(map(len, values)) < 2:
        print(f"Insufficient observations for {feature}")
        return None

    anova = stats.f_oneway(*values)
    tukey = pairwise_tukeyhsd(plot_data[feature], plot_data[GROUP_COL], alpha=0.05)
    letters = tukey_compact_letters(tukey, groups)
    print(f"\n{feature} | ANOVA p = {anova.pvalue:.4g}")
    print(tukey.summary())
    print("Tukey significance letters:", letters)

    fig, ax = plt.subplots(figsize=(max(11, len(groups) * 1.1), 6), dpi=150)
    sns.boxplot(
        data=plot_data, x=GROUP_COL, y=feature, order=groups,
        palette=PALETTE, hue=GROUP_COL, dodge=False, legend=False,
        width=0.6, showfliers=False, showmeans=True,
        meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black"},
        ax=ax,
    )
    sns.stripplot(
        data=plot_data, x=GROUP_COL, y=feature, order=groups,
        color="gray", size=4, jitter=True, alpha=0.7, ax=ax,
    )

    # Position each letter above its group's observed maximum to avoid
    # overlapping data points or outliers.
    y_min, y_max = plot_data[feature].min(), plot_data[feature].max()
    span = max(float(y_max - y_min), abs(float(y_max)) * 0.05, 1e-9)
    for i, group in enumerate(groups):
        group_max = plot_data.loc[plot_data[GROUP_COL] == group, feature].max()
        ax.text(i, group_max + 0.04 * span, letters[group],
                ha="center", va="bottom", fontsize=13, fontweight="bold")
    ax.set_ylim(top=y_max + 0.18 * span)
    ax.set_title(f"{feature} | one-way ANOVA p = {anova.pvalue:.3g}")
    ax.tick_params(axis="x", rotation=60)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{feature}.svg", bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{feature}.png", dpi=300, bbox_inches="tight")
    plt.show()
    (OUTPUT_DIR / f"{feature}_tukey.txt").write_text(str(tukey.summary()), encoding="utf-8")
    pd.DataFrame({"Group": groups, "Tukey_letters": [letters[g] for g in groups]}).to_csv(
        OUTPUT_DIR / f"{feature}_letters.csv", index=False
    )
    return {"Feature": feature, "ANOVA_F": anova.statistic, "ANOVA_p": anova.pvalue}


feature_results = [analyze_feature(feature) for feature in SELECTED_FEATURES]
pd.DataFrame([row for row in feature_results if row]).to_csv(
    OUTPUT_DIR / "individual_feature_anova.csv", index=False
)

## 3. Compute category-level Welch test scores

For each treatment/time group and each feature in a defined category, Welch's independent-samples t-test compares that group against `Non-treated_1h`. A score of `−log10(p)` is computed for each feature and averaged across features in the category. The resulting score summarizes statistical evidence, **not** the magnitude or direction of morphological changes.

No p-value correction is applied in this section to preserve the original calculation. This limitation should be acknowledged when interpreting category-level results.

In [ ]:
def category_scores(data, feature_groups, group_col=GROUP_COL, control=CONTROL_GROUP):
    records = []
    control_df = data.loc[data[group_col] == control]
    for group in [g for g in GROUP_ORDER if g in data[group_col].unique() and g != control]:
        treatment_df = data.loc[data[group_col] == group]
        for category, features in feature_groups.items():
            p_values = []
            for feature in features:
                if feature not in data.columns or not pd.api.types.is_numeric_dtype(data[feature]):
                    continue
                a = treatment_df[feature].dropna().to_numpy()
                b = control_df[feature].dropna().to_numpy()
                if len(a) < 2 or len(b) < 2:
                    continue
                p = ttest_ind(a, b, equal_var=False).pvalue
                if np.isfinite(p): p_values.append(max(p, 1e-300))
            if p_values:
                records.append({"Group": group, "Category": category,
                                "Mean_neg_log10_p": float(np.mean(-np.log10(p_values))),
                                "N_features": len(p_values)})
    return pd.DataFrame(records)

def plot_category_scores(feature_groups, filename, title):
    scores = category_scores(df, feature_groups)
    if scores.empty:
        print(f"No eligible features for {filename}")
        return scores
    wide = scores.pivot(index="Group", columns="Category", values="Mean_neg_log10_p")
    wide = wide.reindex([g for g in GROUP_ORDER if g != CONTROL_GROUP and g in wide.index]).fillna(0)
    fig, ax = plt.subplots(figsize=(max(11, len(wide)*1.1), 6))
    wide.plot(kind="bar", stacked=True, ax=ax, width=.8)
    ax.set(ylabel="Sum of category mean −log10(p) scores", xlabel="Treatment and time", title=title)
    ax.tick_params(axis="x", rotation=60)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{filename}.svg", bbox_inches="tight")
    plt.show()
    scores.to_csv(OUTPUT_DIR / f"{filename}.csv", index=False)
    return scores

## 4. Group morphological features by measurement type

The original analysis grouped features by column-name substrings such as `Texture`, `Granularity`, `AreaShape`, and `Intensity`. Both the three-category and four-category summaries are retained because they were used in the study. The feature-to-category matching is intentionally the same as in the original code.

In [ ]:
measurement_groups = {
    name: [column for column in df.columns if name in column]
    for name in ["Texture", "Granularity", "AreaShape", "Intensity"]
}
plot_category_scores(
    {k: measurement_groups[k] for k in ["Texture", "Granularity", "AreaShape"]},
    "measurement_categories_three", "Morphological feature categories: three groups"
)
plot_category_scores(
    measurement_groups,
    "measurement_categories_four", "Morphological feature categories: including intensity"
)

## 5. Group features by cellular compartment

The original notebook generated summaries for five compartments (`Nuclei`, `Cells`, `Cytoplasm`, `Vesicles`, and `Nucleoli`) and a restricted comparison of nuclei, whole cells, and cytoplasm. Both are preserved. Matching is substring-based and therefore categories can overlap.

In [ ]:
compartment_groups = {
    name: [column for column in df.columns if name in column]
    for name in ["Nuclei", "Cells", "Cytoplasm", "Vesicles", "Nucleoli"]
}
plot_category_scores(
    compartment_groups, "cellular_compartments_five",
    "Morphological evidence by cellular compartment"
)
plot_category_scores(
    {k: compartment_groups[k] for k in ["Nuclei", "Cells", "Cytoplasm"]},
    "cellular_compartments_three", "Nuclear, cellular, and cytoplasmic categories"
)

## 6. Compare fluorescent channel-associated features

Column names containing `GFP` or `PI` are analyzed separately to summarize differences associated with each imaging channel. These scores do not estimate fluorescence intensity directly; they aggregate the statistical evidence across features whose names contain each channel label.

In [ ]:
channel_groups = {
    "GFP": [column for column in df.columns if "GFP" in column],
    "PI": [column for column in df.columns if "PI" in column],
}
plot_category_scores(channel_groups, "fluorescence_channels", "Morphological evidence by fluorescence channel")

## 7. Link to the LDA top-30 feature ranking

The original notebook repeated the LDA feature-ranking code. This is now performed once in Notebook 03, where the fitted LDA model and its feature coefficients are available. Use the `top_30_LDA*.csv` outputs from that notebook instead of refitting LDA here. This prevents inconsistencies between independently fitted models.

In [ ]:
lda_results_dir = Path("../results/lda")
rankings = sorted(lda_results_dir.glob("top_30_LDA*.csv"))
if rankings:
    for file in rankings: print(f"LDA feature ranking available: {file}")
else:
    print("Run Notebook 03 to generate the top-30 LDA feature rankings.")